# C.1 寫出中間步驟能幫忙嗎？


2+3可以直接答5，也可以寫「從2開始數三次：3、4、5，所以是5」。第二份讓讀者核對方法，後續生成也能讀已寫的中間結果；但文字更長，步驟錯了也可能把結論帶錯。先分清回答形式與能力，不把長解釋自動當較可靠。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import ByteTokenizer

tokenizer = ByteTokenizer()
examples = [
    {"question": "2+3=?", "answer": "5"},
    {"question": "2+3=?", "answer": "從2開始數三次：3、4、5，所以是5。"},
]
for example in examples:
    ids = tokenizer.encode(example["answer"])
    print(example["question"], example["answer"], "回答token數", len(ids))

兩份人工答案由ByteTokenizer按UTF-8 byte編號，長1與47，沒有加角色或EOS。這是當前字元編碼下的長度，不是所有tokenizer通用。程式只看材料，沒有訓練或生成。

比較短答與步驟時，需要相同題目範圍、相同起點及新的數字家族，並說明配對的是題數還是token預算。舊 `(a+b)+c` 模型兩支都更新900次，但步驟讀過約9.6倍有效目標，單候選答對11/24，短答1/24；這個差別不能說成等計算量比較。

可見步驟是輸出的文字，不是模型內部計算的完整記錄。它可作為可查解法，也可能流暢卻錯。下一節用最後答對、前一步仍錯的短例，說明為何要另驗每個部分。

<details>
<summary>補充：舊短答／步驟模型的配方與分母</summary>

課程實測真正建立兩個模型比較`(a+b)+c=?`，a、b、c都在0至5之間。相同三個數字的所有排列共用一個家族，216題整組分成167／25／24題、44／6／6個訓練／驗證／最後檢查家族；測的是沒教過的數字組合，不是新數字範圍。兩支從同一份隨機權重複製，寬64、兩層、兩頭，上下文最多128個token位置，包含提問、對話標記與已生成回答，沒有載入SFT或預訓練底座。短答只教最後整數，步驟教`1+2=3;3+3=6;answer=6`這類可驗算的格式。

兩支都以seed 42、每批24題、學習率0.003更新900次；學習率控制每次權重更新的步幅。它們抽到同樣數量的訓練例子，但步驟支線的回答長得多。下表的生成成績各是在同一24題、溫度0.7、每題真的抽一份候選後取得。這裡的[溫度](https://birdhackor.github.io/tiny-perceptron-vlm/1.15.html)是抽樣旋鈕：先把候選分數除以0.7再換成機率，相較於溫度1，機率會更集中於高分候選；它不會改動模型權重。

表中的NLL是負對數似然（negative log-likelihood）。這份報告逐位置給模型正確答案前文，取正確下一token機率的負自然對數，再把這些代價加總、除以有效回答目標數，含EOS；得到的是每個目標的平均代價，計算方法可回看[1.8](https://birdhackor.github.io/tiny-perceptron-vlm/1.8.html)。表格把平均代價與目標數分成兩列，已經平均過的4.3159不需要再除以51。

| 實際紀錄 | 短答 | 可驗算步驟 |
| --- | --- | --- |
| 訓練有效回答目標，含EOS | 48,803 | 466,322 |
| 更新後完整訓練平均NLL | 0.0268 | 0.000262 |
| 完整最後檢查平均NLL | 4.3159 | 0.1372 |
| 最後檢查有效回答目標數，含EOS | 51 | 510 |
| 單候選最後答對 | 1/24 | 11/24 |

步驟版本本輪答對較多，仍錯13題，而且用了約9.6倍的有效訓練文字目標。每份短答最多新增8個token，步驟則最多48個，生成預算也較大。這支持本次配方的有限觀察，沒有完成相同token預算的比較；兩種回答內容不同，NLL也不能當成同一量尺的通用能力分數。同起點核對、固定資料切分與原始生成保留在[正式報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/reasoning.json)。想完整重做時，可在[C.7末尾的選讀操作](https://birdhackor.github.io/tiny-perceptron-vlm/C.7.html)找到入口；先讀懂這個對照，再決定是否花時間訓練。

</details>
